# 📊 圖文比例計算器 v4
**功能：**
- 掛載 Google Drive，自動遞迴掃描所有專案資料夾
- **判斷專案資料夾：資料夾內含 `*_content.txt` 即視為一個專案**（子資料夾如 `*_images/`、`*_videos/` 不影響判斷）
- 每個專案讀取 `*_content.txt` + `*_ocr_content.txt`，合併計算字數
- 讀取 `*_resource_log.csv` 統計圖片與影片數量
- 自動擷取分類路徑（如 `教育/成功`）
- 輸出彙整 CSV 至指定資料夾

**資料夾結構範例：**
```
ZecZec_Group_Data/
└── ZecZec_Dataset_E&G/
    └── 預購式專案/
        └── 教育/
            └── 成功/
                └── ES32_hohoka-01/          ← 含 *_content.txt → 視為專案
                    ├── ES32_images/         ← 子資料夾，忽略不影響判斷
                    ├── ES32_videos/         ← 子資料夾，忽略不影響判斷
                    ├── ES32_content.txt
                    ├── ES32_ocr_content.txt
                    └── ES32_resource_log.csv
```

## ① 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive 已掛載至 /content/drive')

## ② 設定路徑（依實際情況修改）

In [ ]:
import os

# ✏️ 根資料夾：ZecZec_Group_Data 的完整路徑
ROOT_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/ZecZec_Dataset_E&G'

# ✏️ 輸出 CSV 路徑（存回 Drive）
OUTPUT_CSV = '/content/drive/MyDrive/ZecZec_Group_Data/image_text_ratio_result.csv'

print(f'根資料夾：{ROOT_DIR}')
print(f'根資料夾存在：{os.path.isdir(ROOT_DIR)}')
print(f'輸出路徑：{OUTPUT_CSV}')

## ③ 核心函式

In [ ]:
import re
import csv

# ────────────────────────────────────────────
# 1. 字數計算（中文字 + 英文詞，排除標點）
# ────────────────────────────────────────────
PUNCT_RE = re.compile(
    r'['
    r'\u0000-\u001F'
    r'\u0020'
    r'！-／：-＠［-｀｛-～'
    r'\u3000-\u303F'
    r'\uFF00-\uFFEF'
    r'!-/:-@\[-`{-~'
    r']+',
    re.UNICODE
)

def count_words_in_file(path: str) -> int:
    """讀取單一 txt，計算中文字數 + 英文詞數。"""
    if not os.path.isfile(path):
        return 0
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    # 中文字
    zh_count = len(re.findall(r'[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF]', content))
    # 英文詞（移除中文與標點後切分）
    no_zh    = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF]', ' ', content)
    no_punct = PUNCT_RE.sub(' ', no_zh)
    en_count = len([w for w in no_punct.split() if re.search(r'[A-Za-z]', w)])
    return zh_count + en_count

def count_words_in_project(project_path: str) -> tuple[int, list[str]]:
    """
    讀取資料夾內所有符合條件的 txt 檔：
      - *_content.txt（主內文）
      - *_ocr_content.txt（OCR 版）
    回傳合併字數與讀取到的檔名清單。
    """
    total = 0
    found = []
    for fname in os.listdir(project_path):
        # 符合 *_content.txt 或 *_ocr_content.txt
        if fname.endswith('_content.txt') or fname.endswith('_ocr_content.txt'):
            fpath = os.path.join(project_path, fname)
            total += count_words_in_file(fpath)
            found.append(fname)
    return total, found


# ────────────────────────────────────────────
# 2. 媒體統計（讀 *_resource_log.csv）
# ────────────────────────────────────────────
def classify_type(val: str) -> str:
    t = val.strip().lower()
    if t.startswith('image'):
        return 'image'
    elif t == 'video':
        return 'video'
    return 'unknown'

def count_media_in_project(project_path: str) -> tuple[int, int, str | None]:
    """
    尋找資料夾內的 *_resource_log.csv，統計圖片與影片數。
    回傳 (image_count, video_count, log_filename | None)。
    """
    log_file = None
    for fname in os.listdir(project_path):
        if fname.endswith('_resource_log.csv'):
            log_file = fname
            break

    if log_file is None:
        return 0, 0, None

    csv_path   = os.path.join(project_path, log_file)
    img_count  = 0
    vid_count  = 0

    with open(csv_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            return 0, 0, log_file
        fn_lower = {k.strip().lower(): k for k in reader.fieldnames}
        if 'type' not in fn_lower:
            return 0, 0, log_file
        type_col = fn_lower['type']
        for row in reader:
            k = classify_type(row[type_col])
            if k == 'image':
                img_count += 1
            elif k == 'video':
                vid_count += 1

    return img_count, vid_count, log_file


# ────────────────────────────────────────────
# 3. 比例計算
# ────────────────────────────────────────────
def calc_ratio(word_count: int, media_count: int) -> str:
    """每百字有幾筆媒體；字數為 0 回傳 N/A。"""
    if word_count == 0:
        return 'N/A'
    return str(round(media_count / word_count * 100, 2))


# ────────────────────────────────────────────
# 4. 判斷是否為專案資料夾
#    條件：資料夾內含任何 *_content.txt 檔案
#    （有無子資料夾不影響判斷）
# ────────────────────────────────────────────
def is_project_dir(path: str) -> bool:
    """資料夾內含 *_content.txt 即視為專案資料夾。"""
    for fname in os.listdir(path):
        if fname.endswith('_content.txt'):
            return True
    return False


# ────────────────────────────────────────────
# 5. 遞迴掃描，擷取分類路徑
# ────────────────────────────────────────────
def scan_projects(root_dir: str) -> list[dict]:
    """
    遞迴走訪 root_dir，找出所有最深層資料夾。
    category = 從 root_dir 到專案資料夾之間的相對路徑（排除專案本身）。
    """
    results = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        # 排除 root_dir 本身
        if dirpath == root_dir:
            continue

        # 只處理含 *_content.txt 的專案資料夾
        if not is_project_dir(dirpath):
            continue

        project_name = os.path.basename(dirpath)

        # 分類路徑：root → 專案之間的中間層
        rel_path = os.path.relpath(dirpath, root_dir)   # e.g. 預購式專案/教育/成功/ES32_hohoka-01
        parts    = rel_path.replace('\\', '/').split('/')
        category = '/'.join(parts[:-1]) if len(parts) > 1 else ''

        # 字數
        word_count, txt_files = count_words_in_project(dirpath)

        # 媒體數
        img_count, vid_count, log_fname = count_media_in_project(dirpath)
        total_media = img_count + vid_count

        # 警告提示
        if not txt_files:
            print(f'  ⚠️  [{project_name}] 找不到 *_content.txt，字數記為 0')
        if log_fname is None:
            print(f'  ⚠️  [{project_name}] 找不到 *_resource_log.csv，媒體數記為 0')

        ratio = calc_ratio(word_count, total_media)

        results.append({
            'project':             project_name,
            ##'category':            category,
            'word_count':          word_count,
            'image_count':         img_count,
            'video_count':         vid_count,
            'media_per_100_words': ratio,
        })

        print(f'  ✅  [{project_name}]  分類={category}  '
              f'字數={word_count}  圖片={img_count}  '
              f'影片={vid_count}  每百字媒體={ratio}')

    return results


# ────────────────────────────────────────────
# 6. 輸出 CSV
# ────────────────────────────────────────────
FIELDNAMES = ['project', 'word_count',
              'image_count', 'video_count', 'media_per_100_words']

def write_csv(results: list[dict], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'a+', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(results)
    print(f'\n📁 結果已輸出至：{output_path}')

print('✅ 函式載入完成')

## ④ 執行掃描並輸出結果

In [ ]:
print('=' * 60)
print('     📊 圖文比例計算器 v4（Colab + Google Drive）')
print('=' * 60)
print(f'\n🔍 掃描根資料夾：{ROOT_DIR}\n')

results = scan_projects(ROOT_DIR)

if results:
    write_csv(results, OUTPUT_CSV)
    print(f'\n共處理 {len(results)} 個專案')
else:
    print('\n⚠️  沒有找到任何最深層專案資料夾。')

## ⑤ 預覽結果

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
print(f'共 {len(df)} 筆專案')
df